# שיעור 11 - פרוטוקול הקשר מודל (MCP)

פרוטוקול הקשר מודל (**Model Context Protocol (MCP)**) הוא תקן פתוח שמאפשר לסוכנים לגלות ולהשתמש בכלים, משאבים ומקורות נתונים באופן דינמי בזמן ריצה. במקום לקלוע כלים ישירות לתוך סוכן, MCP מאפשר לסוכנים להתחבר לשרתים חיצוניים שמציגים יכולות על פי דרישה.

בשיעור זה תלמדו:
- מהו MCP ולמה הוא חשוב למערכות סוכנים
- איך פועלת ארכיטקטורת לקוח-שרת של MCP
- איך לבנות סוכנים שמשתמשים בגילוי כלים בסגנון MCP


## התקנה

**דרישות מוקדמות:**
- פרויקט Microsoft Foundry עם מודל פרוס
- להריץ `az login` לאימות


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## מה זה פרוטוקול הקשר למודל (MCP)?

MCP מגדיר דרך סטנדרטית לסוכני בינה מלאכותית לגלות ולתקשר עם כלים חיצוניים ומקורות נתונים:

- **שרת MCP**: חושף כלים, משאבים, והנחיות דרך פרוטוקול סטנדרטי
- **לקוח MCP**: זמן ריצה של הסוכן שמתחבר לשרתים ומגלה את היכולות הזמינות
- **גילוי דינמי**: לסוכנים לא נדרשים כלים מקודדים — הם מגלים מה זמין בזמן ריצה

זה עוצמתי לבניית מערכות סוכנים ניתנות להרחבה שבהן אפשר להוסיף יכולות חדשות מבלי לשנות את קוד הסוכן.


## כיצד MCP פועל

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. הסוכן (לקוח MCP) מתחבר לשרת MCP
2. השרת משיב עם רשימה של כלים זמינים ושל הסכימות שלהם
3. הסוכן יכול אז לקרוא לכל כלי שזוהה במהלך החשיבה שלו
4. התוצאות זורמות חזרה דרך אותו הפרוטוקול


## חיקוי גילוי כלי MCP

מכיוון ששרת MCP אמיתי דורש תהליך שרת פעיל, נציג את הדפוס בעזרת פונקציות `@tool` המדמות את מה ששירות אירוח מקושר ל-MCP יספק.

בסביבת ייצור, כלים אלה יתגילו באופן דינמי משרת MCP במקום להיות מוגדרים באופן מקומי.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## בניית סוכן עם כלים בסגנון MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP בסביבת ייצור

בסביבת ייצור, MCP מאפשר דפוסים חזקים:

- **גילוי כלים דינמי**: סוכנים מתחברים לשרתי MCP ומגלים כלים בזמן ריצה
- **ארכיטקטורה מנותקת**: ספקי כלים יכולים לעדכן באופן עצמאי מהסוכן
- **שיתוף בין ארגוני**: צוותים יכולים לחשוף יכולות דרך שרתי MCP שכל סוכן יכול להשתמש בהם
- **תמיכה במסגרת הסוכן של מייקרוסופט**: ל-MAF יש תמיכה מובנית בלקוח MCP דרך האינטגרציה `mcp`

כדי להשתמש בשרת MCP אמיתי עם MAF, תתחבר דרך `hosted_mcp_tool()` או האינטגרציה של לקוח MCP.

**למידע נוסף:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

In this lesson, you learned:
- **MCP** is an open standard for dynamic tool discovery between agents and tool providers
- The **client-server architecture** lets agents discover capabilities at runtime
- MCP enables **extensible, decoupled agent systems** where tools can be added without code changes
- Microsoft Agent Framework provides **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
